### Transformacion de la tabla "smartdata_proyect_bronze.movies"

In [0]:
%run "../../utilerias/funciones_comunes"

In [0]:
%run "../../utilerias/configurationes"

In [0]:
dbutils.widgets.text("param_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("param_file_date")


### Paso 1 - Crear el dataframe de movies

In [0]:
movie_df = spark.read.table("smartdata_proyect_bronze.movies").filter(f"file_date = '{v_file_date}'") 

### 2- Seleccionar columnas solo requeridas

In [0]:
#Seleccionamos solo las columnas requeridas.
movie_df_selected = movie_df.select(
    "movieId", "title", "budget", "popularity",
    "yearReleaseDate", "releaseDate", "revenue",
    "durationTime", "voteAverage", "voteCount",
    "file_date", "enviroment", "ingestion_date"
)
 

### 3 - Cambiar nombre de columnas segun lo requerido

In [0]:
#Opcion1 usar funcion withColumnRenamed.
movie_df_renamed = movie_df_selected \
.withColumnRenamed("movieId", "movie_id") \
.withColumnRenamed("yearReleaseDate", "year_release_date") \
    .withColumnRenamed("releaseDate", "release_date") \
.withColumnRenamed("durationTime", "duration_time") \
.withColumnRenamed("voteAverage", "vote_average") \
.withColumnRenamed("voteCount", "vote_count")  

In [0]:
#Opcion2 usar funcion withColumnsRenamed
movie_df_renamed = movie_df_selected.withColumnsRenamed({"movieId": "movie_id", "yearReleaseDate": "year_release_date", "releaseDate": "release_date", "durationTime": "duration_time", "voteAverage": "vote_average", "voteCount": "vote_count"})  

#### Escribir los datos en una tabla con external location.

In [0]:
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.file_date = src.file_date'
merge_delta_lake("smartdata_proyect_silver","movies",movie_df_renamed, merge_condition, "file_date")

In [0]:
%sql
select file_date, count(1) from smartdata_proyect_silver.movies group by file_date;

In [0]:
%sql
select * from smartdata_proyect_silver.movies where file_date = :param_file_date;

In [0]:
dbutils.notebook.exit("success")